In [21]:
using Plots, LazySets, LinearAlgebra

In [22]:

using MAT
# https://juliaio.github.io/MAT.jl/stable/methods/

filedict = matread("building.mat")

Dict{String, Any} with 8 entries:
  "B"   => [0.0; 0.0; … ; 0.0; 0.0;;]
  "A"   => sparse([25, 26, 27, 28, 29, 30, 31, 32, 33, 34  …  39, 40, 41, 42, 4…
  "w"   => [0.1; 0.125866; … ; 810.699; 1000.0;;]
  "mag" => [1.5852e-5; 1.99556e-5; … ; 1.69106e-5; 1.37051e-5;;]
  "C"   => [0.0 0.0 … 0.0 0.0]
  "S"   => sparse([1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  45, 46, 47, 48, 46, 47, 48…
  "hsv" => [0.0025035; 0.00242849; … ; 6.95314e-9; 6.61879e-9;;]
  "R"   => sparse([1, 1, 2, 1, 2, 3, 1, 2, 3, 4  …  39, 40, 41, 42, 43, 44, 45,…

In [23]:

# System matrix (As a sparse matrix)
systemMatrix = filedict["A"]

# Input matrix
inputMatrix = filedict["B"]

# state domain
X = Universe(48)

# input domain
U = BallInf([0.5], 0.3)

# This follows x' = Ax + Bu, x ∈ X, u ∈ U


BallInf{Float64, Vector{Float64}}([0.5], 0.3)

In [ ]:
# From meta file
using ReachabilityAnalysis

X0 = Hyperrectangle(; low=[fill(0.0002, 10); zeros(14); -0.0001; zeros(23)], high=[fill(0.00025, 10); zeros(14); 0.0001; zeros(23)])


Hyperrectangle{Float64, Vector{Float64}, Vector{Float64}}([0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5, 2.500000000000001e-5  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [30]:
Z = convert(Zonotope, X0)

Zonotope{Float64, Vector{Float64}, Matrix{Float64}}([0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225, 0.000225  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.500000000000001e-5 0.0 … 0.0 0.0; 0.0 2.500000000000001e-5 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0])

In [31]:
dim(Z)

48

In [11]:
using ReachabilityModels
using MAT

In [32]:

problem = load_model("building")


LoadError: LoadError: ArgumentError: Package ReachabilityModels does not have MAT in its dependencies:
- You may have a partially installed environment. Try `Pkg.instantiate()`
  to ensure all packages in the environment are installed.
- Or, if you have ReachabilityModels checked out for development and have
  added MAT as a dependency but haven't updated your primary
  environment's manifest file, try `Pkg.resolve()`.
- Otherwise you may need to report an issue with ReachabilityModels
in expression starting at C:\Users\grace\.julia\packages\ReachabilityModels\sVuKz\src\models\building\building.jl:5

Below is code related to cos wave and testing plotting

In [ ]:
# Find solution
function aprox(T, tΔ, A, P₁, μ)
    N = floor(Int, T/tΔ )

    ANorm = norm(A, Inf)
    α = (exp(ANorm*tΔ)-1-tΔ*ANorm)/norm(P₁, Inf)
    β = (exp(ANorm*tΔ)-1)*μ/ANorm

    ϕ = exp(A*tΔ)

    ϕp = (I+ϕ)/2
    ϕm = (I-ϕ)/2
    gens = hcat(ϕp*P₁.generators, ϕm*P₁.center, ϕm*P₁.generators)

    R₁ = minkowski_sum(Zonotope(ϕp*P₁.center, gens), Zonotope(zeros(2), (α+β)*I(2)))
    R = R₁
    boxes = []
    push!(boxes, R₁)

    ballβ = Zonotope(zeros(2), β*I(2))

    for i in 2:N
        push!(boxes, minkowski_sum(linear_map(ϕ, boxes[i-1]), ballβ))
        R = R∪box_approximation(boxes[i])
        # global R = R∪box_approximation(boxes[i])
    end
    return (boxes, R)
end

In [ ]:
function rectangleFromHBox(corners, offset)
    Shape((getindex.(corners, 1)).+(offset*tΔ), getindex.(corners, 2))
end

function plotApprox(boxes, T, N)
    xs = range(0, T, length=N)
    maxes = []
    mines = []
    box = []

    plot(size=(500,400), dpi=300, thickness_scaling=1)

    for i in 1:(N)
        approxBox = box_approximation(boxes[i])
        corners = vertices_list(boxes[i])
        plot!(rectangleFromHBox(corners, i - 1), c=:blue,lab="")
        push!(maxes, norm(approxBox, Inf))
        push!(mines, low(approxBox, 2))
    end
end

In [ ]:
# Test
tΔ = 0.1


T = 4
N = floor(Int, T/tΔ )
μ = 0.1
A = [-1. 0.; 0. -1.]#UniformScaling(-1.0)
P₁ = Zonotope([0., 1.5], [0.0 0.0; 0.0 0.5])

xs = range(0, T, length=N)


boxes, R = aprox(T, tΔ, A, P₁, μ)
plotApprox(boxes, T, N)

plot!(xs, 1.0 * exp.(-xs), vars=(0, 1), c=:magenta, lab="")
plot!(xs, 2.0 * exp.(-xs), vars=(0, 1), c=:magenta, lab="")

In [ ]:
# Cos wave
tΔ = 0.1
μ = 0.01

A = [cos(tΔ) sin(tΔ); -sin(tΔ) cos(tΔ)]
P₁ = Zonotope([0., 0.0], [0.0 0.0; 0.0 1])

T = 0.5

boxes, R = aprox(T, tΔ, A, P₁, μ)

N = floor(Int, T/tΔ)


In [ ]:
# Temp workplace, figuring out shapes https://docs.juliaplots.org/dev/input_data/#Shapes


someShape = Shape([(0,0), (1,1), (2,0)])

plot(someShape, c=:blue)


In [ ]:
# Unique plot

function rectangleFromHBox2(corners, currentTime, tΔ)
    values = getindex.(corners, 2)
    #ys = getindex.(corners, 2)

    maxVal = maximum(values)
    minVal = minimum(values)

    xs = [currentTime - tΔ, currentTime]
    ys = [maxVal, minVal]


    # Bad code, fix with more Julia experience
    shape = Shape([(xs[2], ys[2]), (xs[1], ys[2]), (xs[1], ys[1]), (xs[2], ys[1])])

    println("\n Min/Max at time $currentTime :")
    println("Min -> $minVal; Max -> $maxVal")

    return shape
end

xs = range(0, T, length=N)
maxes = []
mines = []
box = []

plot(size=(500,400), dpi=300, thickness_scaling=1)

for i in 1:(N)
    approxBox = box_approximation(boxes[i])
    corners = vertices_list(boxes[i])
    plot!(rectangleFromHBox2(corners, i*tΔ, tΔ), c=:blue,lab="")
    push!(maxes, norm(approxBox, Inf))
    push!(mines, low(approxBox, 2))
end



plot!(xs, cos.(xs), vars=(0, 1), c=:magenta, lab="")
plot!(xs, -cos.(xs), vars=(0, 1), c=:magenta, lab="")

In [ ]:
#Unique plot


# function rectangleFromHBox(corners, offset)
#     Shape((getindex.(corners, 1)).+(offset*tΔ), getindex.(corners, 2))
# end

# function plotApprox(boxes, T, N)
#     xs = range(0, T, length=N)
#     maxes = []
#     mines = []
#     box = []

#     plot(size=(500,400), dpi=300, thickness_scaling=1)

#     for i in 1:(N)
#         approxBox = box_approximation(boxes[i])
#         corners = vertices_list(boxes[i])
#         plot!(rectangleFromHBox(corners, i - 1), c=:blue,lab="")
#         push!(maxes, norm(approxBox, Inf))
#         push!(mines, low(approxBox, 2))
#     end
# end


In [ ]:
# Cos wave proof of concept
μ = 0.01
tΔ = 0.2         
T = 3.0         

A = [cos(tΔ)  sin(tΔ);
    -sin(tΔ)  cos(tΔ)]

vector = [0.0  1.0] 

println("Initial: ", vector)

N = floor(Int, T / tΔ)
println("Steps: ", N)

R = vector

println("---")
for i in 1:N
    R = R * A
    println("Step $i → ", R)
end
println("---")

val = cos(T)

println("Corresponding time val: $val")